# Applicability Domain and Out-of-Distribution Detection

## Scientific objective
Compute maximum and kNN Tanimoto similarity, scaffold novelty, descriptor/latent distances where available, uncertainty, AD categories, OOD warnings, and abstention.

## Inputs
- Calibrated endpoint bundles
- Held-out scaffold test molecules

## Expected outputs
- `results/applicability_domain/test_ad_predictions.csv`
- AD coverage/error summaries
- OOD status distribution

## Dependencies
RDKit, calibrated bundles

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
AD thresholds are predefined in configuration and must be sensitivity-tested per endpoint. A novel scaffold alone warns but does not automatically force abstention unless another rule is met.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Similarity thresholds are representation-specific. Domain status is a model-support assessment, not a statement of chemical impossibility.

## Next notebook
[18_activity_cliff_analysis.ipynb](./18_activity_cliff_analysis.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [ ]:
from toxicity_screening.inference import load_bundles, predict_smiles
bundles={b.endpoint:b for b in load_bundles(ROOT/"models/calibrated")}

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

test=records[(records.scaffold_split=="test") & records.label.notna()].copy()
cap=400 if PROFILE=="smoke" else len(test)
rows=[]
for endpoint, group in test.groupby("endpoint"):
    sample=group.sample(min(len(group),cap),random_state=SEED)
    for row in sample.itertuples(index=False):
        out=predict_smiles(row.standardized_smiles,[bundles[endpoint]],str(row.molecule_id)).iloc[0].to_dict(); out["true_label"]=int(row.label); rows.append(out)
ad=pd.DataFrame(rows); ad.to_csv(ROOT/"results/applicability_domain/test_ad_predictions.csv",index=False)
summary=ad.groupby(["endpoint","applicability_domain"]).agg(n=("molecule_id","size"),supported=("abstention_status",lambda x:int((x=="supported").sum())),accuracy=("predicted_class",lambda x:float(np.mean(pd.to_numeric(x,errors="coerce")==ad.loc[x.index,"true_label"])))).reset_index()
summary.to_csv(ROOT/"results/applicability_domain/ad_performance_summary.csv",index=False); display(summary)

In [3]:
# Compare complementary AD/OOD diagnostics using memory-bounded,
# elementwise numerical operations.

import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from toxicity_screening.neural_models import FingerprintMLP
from toxicity_screening.scaffolds import scaffold_novelty


# ------------------------------------------------------------------
# Runtime safety
# ------------------------------------------------------------------

for variable in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")

torch.set_num_threads(1)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    # PyTorch may reject this after its thread pool has been initialized.
    pass


# ------------------------------------------------------------------
# Diagonal-shrinkage Mahalanobis domain
# ------------------------------------------------------------------

class SafeDiagonalDistanceDomain:
    """
    Memory-safe diagonal approximation to a shrinkage Mahalanobis domain.

    This avoids full covariance inversion and BLAS-backed quadratic forms.
    """

    def __init__(
        self,
        quantile_inside: float = 0.95,
        quantile_borderline: float = 0.99,
        shrinkage: float = 0.10,
        variance_floor: float = 1e-10,
    ):
        self.quantile_inside = float(quantile_inside)
        self.quantile_borderline = float(quantile_borderline)
        self.shrinkage = float(shrinkage)
        self.variance_floor = float(variance_floor)

        self.location_ = None
        self.variance_ = None
        self.inside_threshold_ = None
        self.borderline_threshold_ = None

    def fit(self, values: np.ndarray):
        values = np.asarray(
            values,
            dtype=np.float64,
        )

        if values.ndim != 2:
            raise ValueError(
                f"Expected a 2D matrix, received {values.shape}"
            )

        if values.shape[0] < 2:
            raise ValueError(
                "At least two training records are required"
            )

        self.location_ = np.mean(
            values,
            axis=0,
            dtype=np.float64,
        )

        variance = np.var(
            values,
            axis=0,
            ddof=1,
            dtype=np.float64,
        )

        positive_variance = variance[
            np.isfinite(variance)
            & (variance > self.variance_floor)
        ]

        if len(positive_variance):
            variance_target = float(
                np.median(positive_variance)
            )
        else:
            variance_target = 1.0

        variance = (
            (1.0 - self.shrinkage) * variance
            + self.shrinkage * variance_target
        )

        variance = np.where(
            np.isfinite(variance)
            & (variance > self.variance_floor),
            variance,
            variance_target,
        )

        self.variance_ = variance

        training_distance = self.distance(values)

        self.inside_threshold_ = float(
            np.quantile(
                training_distance,
                self.quantile_inside,
            )
        )

        self.borderline_threshold_ = float(
            np.quantile(
                training_distance,
                self.quantile_borderline,
            )
        )

        return self

    def distance(self, values: np.ndarray) -> np.ndarray:
        if (
            self.location_ is None
            or self.variance_ is None
        ):
            raise RuntimeError(
                "Distance domain has not been fitted"
            )

        values = np.asarray(
            values,
            dtype=np.float64,
        )

        difference = values - self.location_

        squared_standardized = (
            difference * difference
        ) / self.variance_

        return np.sqrt(
            np.sum(
                squared_standardized,
                axis=1,
                dtype=np.float64,
            )
        )

    def status(self, values: np.ndarray) -> np.ndarray:
        distance = self.distance(values)

        return np.where(
            distance <= self.inside_threshold_,
            "inside",
            np.where(
                distance
                <= self.borderline_threshold_,
                "borderline",
                "outside",
            ),
        )


# ------------------------------------------------------------------
# Exact but chunked kNN Tanimoto similarity
# ------------------------------------------------------------------

def chunked_knn_tanimoto(
    query: np.ndarray,
    reference: np.ndarray,
    k: int = 5,
    query_chunk_size: int = 8,
    reference_chunk_size: int = 128,
) -> dict[str, np.ndarray]:
    """
    Calculate exact top-k Tanimoto similarities without sklearn's
    pairwise-distance implementation or large full matrices.
    """

    query = np.asarray(
        query,
        dtype=np.uint8,
    )

    reference = np.asarray(
        reference,
        dtype=np.uint8,
    )

    if query.ndim != 2 or reference.ndim != 2:
        raise ValueError(
            "Query and reference fingerprints must be 2D"
        )

    if query.shape[1] != reference.shape[1]:
        raise ValueError(
            "Query and reference fingerprint dimensions differ"
        )

    if len(reference) == 0:
        raise ValueError(
            "Reference fingerprint matrix is empty"
        )

    effective_k = min(
        int(k),
        len(reference),
    )

    query_on_bits = np.sum(
        query,
        axis=1,
        dtype=np.int32,
    )

    reference_on_bits = np.sum(
        reference,
        axis=1,
        dtype=np.int32,
    )

    mean_similarity = np.empty(
        len(query),
        dtype=np.float64,
    )

    minimum_similarity = np.empty(
        len(query),
        dtype=np.float64,
    )

    for query_start in range(
        0,
        len(query),
        query_chunk_size,
    ):
        query_stop = min(
            query_start + query_chunk_size,
            len(query),
        )

        query_batch = query[
            query_start:query_stop
        ]

        query_batch_counts = query_on_bits[
            query_start:query_stop
        ]

        best_similarity = np.full(
            (
                len(query_batch),
                effective_k,
            ),
            -np.inf,
            dtype=np.float64,
        )

        for reference_start in range(
            0,
            len(reference),
            reference_chunk_size,
        ):
            reference_stop = min(
                reference_start
                + reference_chunk_size,
                len(reference),
            )

            reference_batch = reference[
                reference_start:reference_stop
            ]

            reference_batch_counts = (
                reference_on_bits[
                    reference_start:reference_stop
                ]
            )

            # The temporary Boolean tensor remains small because both
            # query and reference batches are explicitly bounded.
            intersection = np.count_nonzero(
                np.bitwise_and(
                    query_batch[:, None, :],
                    reference_batch[None, :, :],
                ),
                axis=2,
            ).astype(np.float64)

            union = (
                query_batch_counts[:, None]
                + reference_batch_counts[None, :]
                - intersection
            )

            similarities = np.divide(
                intersection,
                union,
                out=np.ones_like(
                    intersection,
                    dtype=np.float64,
                ),
                where=union > 0,
            )

            candidates = np.concatenate(
                [
                    best_similarity,
                    similarities,
                ],
                axis=1,
            )

            best_similarity = np.partition(
                candidates,
                candidates.shape[1]
                - effective_k,
                axis=1,
            )[:, -effective_k:]

        mean_similarity[
            query_start:query_stop
        ] = np.mean(
            best_similarity,
            axis=1,
        )

        minimum_similarity[
            query_start:query_stop
        ] = np.min(
            best_similarity,
            axis=1,
        )

    return {
        "mean_knn_similarity": mean_similarity,
        "minimum_knn_similarity": minimum_similarity,
    }


# ------------------------------------------------------------------
# Batched MLP latent-embedding extraction
# ------------------------------------------------------------------

def encode_fingerprints(
    model,
    fingerprint_matrix: np.ndarray,
    rows: np.ndarray,
    batch_size: int = 256,
) -> np.ndarray:
    rows = np.asarray(
        rows,
        dtype=np.int64,
    )

    embeddings = []

    model.eval()

    with torch.no_grad():
        for start in range(
            0,
            len(rows),
            batch_size,
        ):
            stop = min(
                start + batch_size,
                len(rows),
            )

            batch = torch.from_numpy(
                np.asarray(
                    fingerprint_matrix[
                        rows[start:stop]
                    ],
                    dtype=np.float32,
                )
            )

            embedding = (
                model.encoder(batch)
                .detach()
                .cpu()
                .numpy()
            )

            embeddings.append(embedding)

    if not embeddings:
        raise ValueError(
            "No fingerprints were supplied for encoding"
        )

    return np.concatenate(
        embeddings,
        axis=0,
    )


# ------------------------------------------------------------------
# Load and validate data
# ------------------------------------------------------------------

archive = np.load(
    ROOT / "data/processed/morgan_features.npz",
    allow_pickle=False,
)

X = archive["X"].astype(
    np.uint8,
    copy=False,
)

ids = archive["molecule_id"].astype(str)

if X.shape[0] != len(ids):
    raise ValueError(
        f"Morgan feature/identifier mismatch: "
        f"{X.shape[0]} rows versus {len(ids)} IDs"
    )

feature_index = pd.DataFrame(
    {
        "molecule_id": ids,
        "row": np.arange(
            len(ids),
            dtype=np.int64,
        ),
    }
)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
).merge(
    feature_index,
    on="molecule_id",
    validate="many_to_one",
)

descriptors = pd.read_csv(
    ROOT / "data/processed/descriptors.csv",
    low_memory=False,
)

descriptor_columns = [
    column
    for column in descriptors.select_dtypes(
        include=[np.number]
    ).columns
    if column not in {
        "row",
        "descriptor_row",
    }
]

if not descriptor_columns:
    raise ValueError(
        "No numerical descriptor columns were found"
    )

descriptor_frame = (
    descriptors[descriptor_columns]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

descriptor_frame = descriptor_frame.fillna(
    descriptor_frame.median(
        numeric_only=True
    )
)

descriptor_values = descriptor_frame.to_numpy(
    dtype=np.float64,
)

descriptor_index = descriptors[
    ["molecule_id"]
].copy()

descriptor_index["molecule_id"] = (
    descriptor_index["molecule_id"].astype(str)
)

descriptor_index["descriptor_row"] = np.arange(
    len(descriptor_index),
    dtype=np.int64,
)

records = records.merge(
    descriptor_index,
    on="molecule_id",
    how="left",
    validate="many_to_one",
)

comparison_rows = []

sample_cap = PROFILE_CONFIG.get(
    "sample_cap_per_endpoint"
)

reference_cap = (
    int(sample_cap)
    if PROFILE == "smoke"
    and sample_cap
    else None
)

test_cap = 400 if PROFILE == "smoke" else None


# ------------------------------------------------------------------
# Endpoint-level AD calculations
# ------------------------------------------------------------------

for endpoint, endpoint_frame in records.loc[
    records["label"].notna()
].groupby(
    "endpoint",
    sort=False,
):
    training = (
        endpoint_frame.loc[
            endpoint_frame["scaffold_split"]
            == "train"
        ]
        .drop_duplicates("molecule_id")
        .copy()
    )

    test = (
        endpoint_frame.loc[
            endpoint_frame["scaffold_split"]
            == "test"
        ]
        .drop_duplicates("molecule_id")
        .copy()
    )

    training = training.loc[
        training["descriptor_row"].notna()
    ].copy()

    test = test.loc[
        test["descriptor_row"].notna()
    ].copy()

    if training.empty or test.empty:
        raise ValueError(
            f"{endpoint}: no valid training or test records"
        )

    if (
        reference_cap is not None
        and len(training) > reference_cap
    ):
        reference = training.sample(
            n=reference_cap,
            random_state=SEED,
        )
    else:
        reference = training

    if (
        test_cap is not None
        and len(test) > test_cap
    ):
        test = test.sample(
            n=test_cap,
            random_state=SEED,
        )

    reference = reference.reset_index(
        drop=True
    )

    test = test.reset_index(
        drop=True
    )

    reference_rows = (
        reference["row"]
        .astype(int)
        .to_numpy()
    )

    test_rows = (
        test["row"]
        .astype(int)
        .to_numpy()
    )

    reference_descriptor_rows = (
        reference["descriptor_row"]
        .astype(int)
        .to_numpy()
    )

    test_descriptor_rows = (
        test["descriptor_row"]
        .astype(int)
        .to_numpy()
    )

    print(
        f"[AD] endpoint_started "
        f"endpoint={endpoint} "
        f"reference_rows={len(reference)} "
        f"test_rows={len(test)}",
        flush=True,
    )

    # Descriptor domain
    descriptor_domain = (
        SafeDiagonalDistanceDomain()
        .fit(
            descriptor_values[
                reference_descriptor_rows
            ]
        )
    )

    descriptor_distance = (
        descriptor_domain.distance(
            descriptor_values[
                test_descriptor_rows
            ]
        )
    )

    descriptor_status = (
        descriptor_domain.status(
            descriptor_values[
                test_descriptor_rows
            ]
        )
    )

    print(
        f"[AD] descriptor_completed "
        f"endpoint={endpoint}",
        flush=True,
    )

    # Fingerprint kNN similarity
    knn = chunked_knn_tanimoto(
        X[test_rows],
        X[reference_rows],
        k=5,
        query_chunk_size=8,
        reference_chunk_size=128,
    )

    print(
        f"[AD] knn_completed "
        f"endpoint={endpoint}",
        flush=True,
    )

    # Latent-space domain
    latent_distance = np.full(
        len(test),
        np.nan,
        dtype=np.float64,
    )

    latent_status = np.full(
        len(test),
        "unavailable",
        dtype=object,
    )

    checkpoint = (
        ROOT
        / "models"
        / "neural"
        / f"{endpoint}_mlp.pt"
    )

    if checkpoint.exists():
        model = FingerprintMLP(
            X.shape[1],
            **CONFIGS[
                "model_config"
            ][
                "neural"
            ][
                "fingerprint_mlp"
            ],
        )

        checkpoint_payload = torch.load(
            checkpoint,
            map_location="cpu",
            weights_only=True,
        )

        if (
            isinstance(
                checkpoint_payload,
                dict,
            )
            and "state_dict"
            in checkpoint_payload
        ):
            state_dict = checkpoint_payload[
                "state_dict"
            ]
        else:
            state_dict = checkpoint_payload

        model.load_state_dict(
            state_dict
        )

        reference_embedding = (
            encode_fingerprints(
                model,
                X,
                reference_rows,
                batch_size=256,
            )
        )

        test_embedding = encode_fingerprints(
            model,
            X,
            test_rows,
            batch_size=256,
        )

        latent_domain = (
            SafeDiagonalDistanceDomain()
            .fit(reference_embedding)
        )

        latent_distance = (
            latent_domain.distance(
                test_embedding
            )
        )

        latent_status = (
            latent_domain.status(
                test_embedding
            )
        )

        del (
            model,
            checkpoint_payload,
            state_dict,
            reference_embedding,
            test_embedding,
            latent_domain,
        )

        gc.collect()

        print(
            f"[AD] latent_completed "
            f"endpoint={endpoint}",
            flush=True,
        )
    else:
        print(
            f"[AD] latent_skipped "
            f"endpoint={endpoint} "
            f"reason=checkpoint_missing",
            flush=True,
        )

    training_scaffolds = set(
        reference["scaffold"]
        .fillna("")
        .astype(str)
    )

    for position, row in enumerate(
        test.itertuples(index=False)
    ):
        comparison_rows.append(
            {
                "endpoint": endpoint,
                "molecule_id": row.molecule_id,
                "reference_n": int(
                    len(reference)
                ),
                "mean_knn_similarity": float(
                    knn[
                        "mean_knn_similarity"
                    ][position]
                ),
                "minimum_knn_similarity": float(
                    knn[
                        "minimum_knn_similarity"
                    ][position]
                ),
                "descriptor_distance": float(
                    descriptor_distance[
                        position
                    ]
                ),
                "descriptor_ad": str(
                    descriptor_status[
                        position
                    ]
                ),
                "latent_distance": float(
                    latent_distance[
                        position
                    ]
                ),
                "latent_ad": str(
                    latent_status[
                        position
                    ]
                ),
                "scaffold_novelty": bool(
                    scaffold_novelty(
                        str(row.scaffold),
                        training_scaffolds,
                    )
                ),
            }
        )

    print(
        f"[AD] endpoint_completed "
        f"endpoint={endpoint}",
        flush=True,
    )

    gc.collect()


# ------------------------------------------------------------------
# Persist and summarize
# ------------------------------------------------------------------

comparison = pd.DataFrame(
    comparison_rows
)

output_path = (
    ROOT
    / "results"
    / "applicability_domain"
    / "ad_method_comparison.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

comparison.to_csv(
    output_path,
    index=False,
)

comparison_summary = (
    comparison.groupby(
        "endpoint",
        as_index=False,
    )
    .agg(
        descriptor_outside=(
            "descriptor_ad",
            lambda values: float(
                (
                    values == "outside"
                ).mean()
            ),
        ),
        latent_available=(
            "latent_ad",
            lambda values: float(
                (
                    values != "unavailable"
                ).mean()
            ),
        ),
        mean_knn_similarity=(
            "mean_knn_similarity",
            "mean",
        ),
        reference_n=(
            "reference_n",
            "first",
        ),
        test_n=(
            "molecule_id",
            "size",
        ),
    )
)

print(
    f"[AD] completed "
    f"rows={len(comparison)} "
    f"output={output_path}",
    flush=True,
)

display(comparison_summary)

[AD] knn_completed endpoint=herg_blockade
[AD] latent_completed endpoint=herg_blockade
[AD] endpoint_completed endpoint=herg_blockade
[AD] endpoint_started endpoint=ames_mutagenicity reference_rows=5015 test_rows=1120
[AD] descriptor_completed endpoint=ames_mutagenicity
[AD] knn_completed endpoint=ames_mutagenicity
[AD] latent_completed endpoint=ames_mutagenicity
[AD] endpoint_completed endpoint=ames_mutagenicity
[AD] endpoint_started endpoint=SR-p53 reference_rows=4811 test_rows=986
[AD] descriptor_completed endpoint=SR-p53
[AD] knn_completed endpoint=SR-p53
[AD] latent_completed endpoint=SR-p53
[AD] endpoint_completed endpoint=SR-p53
[AD] endpoint_started endpoint=SR-ATAD5 reference_rows=5013 test_rows=1027
[AD] descriptor_completed endpoint=SR-ATAD5
[AD] knn_completed endpoint=SR-ATAD5
[AD] latent_completed endpoint=SR-ATAD5
[AD] endpoint_completed endpoint=SR-ATAD5
[AD] endpoint_started endpoint=SR-ARE reference_rows=4139 test_rows=847
[AD] descriptor_completed endpoint=SR-ARE
[AD]

,endpoint,descriptor_outside,latent_available,mean_knn_similarity,reference_n,test_n
0,SR-ARE,0.005903,1.0,0.403951,4139,847
1,SR-ATAD5,0.005842,1.0,0.407931,5013,1027
2,SR-MMP,0.010689,1.0,0.399530,4130,842
3,SR-p53,0.009128,1.0,0.406003,4811,986
4,ames_mutagenicity,0.012500,1.0,0.442620,5015,1120
5,herg_blockade,0.013277,1.0,0.532595,8971,1883


### Completion gate
Confirm that the declared artifacts exist before continuing to `18_activity_cliff_analysis.ipynb`.